# matrixlayout Binder demo

`matrixlayout` builds LaTeX/TikZ layouts for matrix-centered linear algebra figures and renders them to SVG. The examples below are adapted from the `la_figures` Gaussian-elimination, QR/Gram-Schmidt, eigenproblem, and back-substitution examples, but are written directly against the current `matrixlayout` API.

In [ ]:
from matrixlayout import (
    backsubst_svg,
    decorator_bg,
    decorator_box,
    decorator_color,
    render_eig_svg,
    render_ge_svg,
    render_qr_svg,
    sel_col,
    sel_entry,
    show_svg,
)

RENDER_OPTS = dict(toolchain_name="pdftex_dvisvgm", crop="tight", padding=(2, 2, 2, 2))

## Row reduction of an augmented system

This is the row-reduction example used in the `la_figures` Gaussian-elimination notebooks: start with `Ax=b`, expose the zero row, and then show a reduced form with one free variable.

In [ ]:
A0 = [[2, 1, 1], [2, 1, 1], [4, 3, 3]]
b0 = [[5], [5], [11]]

A1 = [[2, 1, 1], [0, 0, 0], [0, 1, 1]]
b1 = [[5], [0], [1]]

A2 = [[1, 0, 0], [0, 1, 1], [0, 0, 0]]
b2 = [[2], [1], [0]]

row_reduction_svg = render_ge_svg(
    matrices=[[A0, b0], [A1, b1], [A2, b2]],
    label_rows=[
        {"grid": (0, 0), "side": "above", "rows": [["A"]]},
        {"grid": (0, 1), "side": "above", "rows": [["b"]]},
        {"grid": (1, 0), "side": "above", "rows": [["U"]]},
        {"grid": (1, 1), "side": "above", "rows": [["c"]]},
        {"grid": (2, 0), "side": "above", "rows": [["R"]]},
        {"grid": (2, 1), "side": "above", "rows": [["d"]]},
    ],
    outer_hspace_mm=8,
    **RENDER_OPTS,
)
show_svg(row_reduction_svg)

## Pivot columns, equation labels, and callouts

The same reduced system is annotated with row and column labels. This is the kind of figure used to discuss pivot variables, free variables, and consistency.

In [ ]:
label_svg = render_ge_svg(
    matrices=[A2],
    label_rows=[{"grid": (0, 0), "side": "above", "rows": [["x1", "x2", "x3"]]}],
    label_cols=[{"grid": (0, 0), "side": "left", "cols": [["p1"], ["p2"], ["0=0"]]}],
    decorations=[
        {"grid": (0, 0), "entries": [sel_col(0)], "decorator": decorator_bg("green!15")},
        {"grid": (0, 0), "entries": [sel_col(1)], "decorator": decorator_bg("green!15")},
        {"grid": (0, 0), "entries": [sel_entry(1, 2)], "decorator": decorator_box(color="red")},
        {"grid": (0, 0), "label": r"x_3	ext{ is free}", "side": "right", "angle": -35, "length": 10},
    ],
    create_medium_nodes=True,
    **RENDER_OPTS,
)
show_svg(label_svg)

## Entry decorators for elimination structure

Selectors and decorators make it possible to call out pivots, eliminated entries, and active columns without manually rewriting the matrix.

In [ ]:
decorated_svg = render_ge_svg(
    matrices=[A1],
    decorations=[
        {"grid": (0, 0), "entries": [sel_entry(0, 0), sel_entry(2, 1)], "decorator": decorator_box(color="blue")},
        {"grid": (0, 0), "entries": [sel_entry(1, 0), sel_entry(1, 1), sel_entry(1, 2)], "decorator": decorator_color("gray")},
        {"grid": (0, 0), "entries": [sel_col(2)], "decorator": decorator_bg("yellow!25")},
    ],
    **RENDER_OPTS,
)
show_svg(decorated_svg)

## Gram-Schmidt / QR layout

This example follows the `la_figures` QR notebook pattern: put the original columns next to the orthonormal columns produced by Gram-Schmidt.

In [ ]:
A = [[1, 1], [1, 0], [0, 1]]
Q = [
    [r"1/\sqrt{2}", r"1/\sqrt{6}"],
    [r"1/\sqrt{2}", r"-1/\sqrt{6}"],
    [0, r"2/\sqrt{6}"],
]
R = [[r"\sqrt{2}", r"1/\sqrt{2}"], [0, r"\sqrt{6}/2"]]

qr_svg = render_qr_svg(
    matrices=[[None, A, Q, R]],
    array_names=[None, "A", "Q", "R"],
    **RENDER_OPTS,
)
show_svg(qr_svg)

## Eigenproblem table

The eigenproblem table is a compact way to show eigenvalues, eigenspace bases, and an eigenbasis matrix for a diagonalizable example.

In [ ]:
eig_spec = {
    "lambda": [2, 3],
    "ma": [1, 1],
    "evecs": [[[1, 0]], [[0, 1]]],
    "qvecs": [[[1, 0]], [[0, 1]]],
}

eig_svg = render_eig_svg(eig_spec, case="S", **RENDER_OPTS)
show_svg(eig_svg)

## SVD-style table

The same table renderer also handles an SVD-style spec with singular values, right singular vectors, and left singular vectors.

In [ ]:
svd_spec = {
    "lambda": [4, 1],
    "ma": [1, 1],
    "sigma": [2, 1],
    "evecs": [[[1, 0]], [[0, 1]]],
    "qvecs": [[[1, 0]], [[0, 1]]],
    "uvecs": [[[1, 0, 0]], [[0, 1, 0]], [[0, 0, 1]]],
    "sz": (3, 2),
}

svd_svg = render_eig_svg(svd_spec, case="SVD", **RENDER_OPTS)
show_svg(svd_svg)

## Back-substitution trace

This mirrors the `la_figures` back-substitution notebook: render the system, the substitution cascade, and the parametric solution together.

In [ ]:
backsubst_demo_svg = backsubst_svg(
    system_txt=r"$x + 2y = 4,\quad y + 3z = 5$",
    cascade_txt=[r"$y = 5 - 3z$", r"$x = 4 - 2(5 - 3z) = 6z - 6$"],
    solution_txt=r"$(x,y,z) = (6t - 6,\; 5 - 3t,\; t)$",
    **RENDER_OPTS,
)
show_svg(backsubst_demo_svg)

## Render options

`toolchain_name`, `crop`, and `padding` are forwarded through the rendering boundary. This notebook uses `pdftex_dvisvgm` because it is compact and reliable in Binder.

In [ ]:
compact_svg = render_ge_svg(
    matrices=[[[1, 2], [3, 4]]],
    toolchain_name="pdftex_dvisvgm",
    crop="tight",
    padding=(0, 0, 0, 0),
)
show_svg(compact_svg)

## Optional: inspect SVG text

The rendered image is the useful output for most notebook work. Inspect raw SVG only when debugging, saving, or comparing renderer output.

In [ ]:
print(backsubst_demo_svg[:500])

## Troubleshooting

If rendering fails, check that the TeX packages and converters are available. The Binder image checks required TeX packages during `postBuild`, and the renderers keep failure artifacts in a temporary directory when a toolchain fails.

In [ ]:
import shutil

checks = ["latex", "dvisvgm", "kpsewhich"]
{name: shutil.which(name) for name in checks}